In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize
import time

print("Initializing Kikuyu Wikipedia Crawler...")

# The 'Special:Random' URL automatically redirects to a random Wikipedia article
random_url = "https://ki.wikipedia.org/wiki/Special:Random"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

kikuyu_sentences = set() # Using a set automatically prevents duplicate sentences

# Keep crawling until we hit exactly 500 sentences
while len(kikuyu_sentences) < 500:
    try:
        # Fetch a random Wikipedia article in Kikuyu
        response = requests.get(random_url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Extract all paragraph text
        for p in soup.find_all('p'):
            text = p.get_text(strip=True)
            
            # If the paragraph is empty, skip it
            if not text:
                continue
                
            # Chop the paragraph into individual sentences
            sentences = sent_tokenize(text)
            
            for sentence in sentences:
                # Quality Control: keep sentences of a reasonable length
                if 30 < len(sentence) < 150:
                    kikuyu_sentences.add(sentence)
                    
                    # Stop exactly at 500
                    if len(kikuyu_sentences) >= 500:
                        break
            
            # Break the outer loop if we reached 500
            if len(kikuyu_sentences) >= 500:
                break
                
        # Being polite to Wikipedia's servers by pausing briefly between page loads
        time.sleep(0.5)
        print(f"Collected {len(kikuyu_sentences)}/500 sentences...", end='\r')
        
    except Exception as e:
        print(f"Skipped a page due to an error: {e}")

# Format the data to match the Rubric
df_kikuyu = pd.DataFrame({
    'Text': list(kikuyu_sentences),
    'Language': 'Kikuyu'
})

print(f"\n\nSuccess! Extracted {len(df_kikuyu)} Kikuyu sentences.")
print(df_kikuyu.head())

# Save this to its own CSV
df_kikuyu.to_csv('kikuyu_data.csv', index=False, encoding='utf-8')
print("\nData saved to 'kikuyu_data.csv'")

Initializing Kikuyu Wikipedia Crawler...
Collected 500/500 sentences...

Success! Extracted 500 Kikuyu sentences.
                                                Text Language
0             Ponte dell'Oliocity irĩa nene yaItaly.   Kikuyu
1  Mũndũ o wothe nĩ akenagio nĩ mwana wĩna mĩtugo...   Kikuyu
2  [1]na imwe nĩ ciĩheaga irio cia kũhaarĩria ikũyũ.   Kikuyu
3      Andu aya ni David wineland na Serge Haroche .   Kikuyu
4  Imwe ciambicaanathaka nĩ hamwe na '我是誰'(Niĩ nd...   Kikuyu

Data saved to 'kikuyu_data.csv'
